# Knowledge Distillation: Learning from a Teacher's Distribution

The full code of [Knowledge Distillation: Learning from a Teacher's Distribution](https://residual-stream.lvtan71.workers.dev/posts/knowledge-distillation/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/knowledge-distillation) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/knowledge-distillation.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 31 min end to end on the RTX 3090 machine that saved the outputs below.
Trains Qwen2.5-0.5B nine times (400 steps each) against Qwen2.5-1.5B-Instruct. The tables are made by two short scripts the post doesn't show, included here.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-18** (torch 2.14.1+cu130, 30 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "torch==2.14.*" "transformers==5.18.*" huggingface_hub pyarrow "numpy==2.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "knowledge-distillation"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/knowledge-distillation


## `dark_knowledge.py`

The teacher's next-token probabilities after one sentence of the blog, at temperature 1 and 2.

In [2]:
!{sys.executable} dark_knowledge.py

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (531066 > 131072). Running this sequence through the model will result in indexing errors


'A language model generates its answer one'


temperature 1: ' word' 0.481, ' character' 0.275, ' sentence' 0.079, ' token' 0.078, ' letter' 0.039, ' step' 0.006, ' symbol' 0.005, ' by' 0.005; top 8 hold 97%
temperature 2: ' word' 0.074, ' character' 0.056, ' sentence' 0.030, ' token' 0.030, ' letter' 0.021, ' step' 0.008, ' symbol' 0.008, ' by' 0.007; top 8 hold 23%


The post printed:

```text
'A language model generates its answer one'
temperature 1: ' word' 0.481, ' character' 0.275, ' sentence' 0.079, ' token' 0.078, ' letter' 0.039, ' step' 0.006, ' symbol' 0.005, ' by' 0.005; top 8 hold 97%
temperature 2: ' word' 0.074, ' character' 0.056, ' sentence' 0.030, ' token' 0.030, ' letter' 0.021, ' step' 0.008, ' symbol' 0.008, ' by' 0.007; top 8 hold 23%
```

## `tau_squared.py`

The soft loss's gradient on ten classes at temperatures 1 to 16, with and without the τ² factor, against the large-τ formula.

In [3]:
!{sys.executable} tau_squared.py

 tau  |gradient|   x tau^2   large-tau formula
   1     0.78426    0.7843             0.89227
   2     0.21055    0.8422             0.22307
   4     0.05381    0.8610             0.05577
   8     0.01366    0.8739             0.01394
  16     0.00345    0.8823             0.00349


The post printed:

```text
 tau  |gradient|   x tau^2   large-tau formula
   1     0.78426    0.7843             0.89227
   2     0.21055    0.8422             0.22307
   4     0.05381    0.8610             0.05577
   8     0.01366    0.8739             0.01394
  16     0.00345    0.8823             0.00349
```

## `forward_reverse_kl.py`

One Gaussian fitted to a two-bump teacher by forward and by reverse KL, on a grid.

In [4]:
!{sys.executable} forward_reverse_kl.py

forward KL(p || q): best q has mean +0.00, width 2.10; mass in the gap |x| < 0.5: 19% (teacher: 1%)
reverse KL(q || p): best q has mean -2.00, width 0.60; mass in the gap |x| < 0.5: 1% (teacher: 1%)


The post printed:

```text
forward KL(p || q): best q has mean +0.00, width 2.10; mass in the gap |x| < 0.5: 19% (teacher: 1%)
reverse KL(q || p): best q has mean -2.00, width 0.60; mass in the gap |x| < 0.5: 1% (teacher: 1%)
```

## `three_labels.py`

Runs `teacher.py` for seeds 0 and 1 and `distill.py` for the three kinds of label and two seeds, keeping each run's summary line in `teacher.log` and `distill.log`, then prints the table with `distill_table.py`.

In [5]:
!{sys.executable} three_labels.py

teacher: blog 1.038, wiki 0.873 bits per byte; its top 64 tokens hold 92.5% of its probability on average


hard lr 1e-05 seed 0 (155 s): step 0: blog 1.163, wiki 0.981 | step 50: blog 1.086, wiki 1.002 | step 100: blog 1.072, wiki 1.013 | step 200: blog 1.071, wiki 1.031 | step 400: blog 1.069, wiki 1.048


soft lr 1e-05 seed 0 (154 s): step 0: blog 1.163, wiki 0.981 | step 50: blog 1.166, wiki 0.991 | step 100: blog 1.165, wiki 0.995 | step 200: blog 1.161, wiki 0.998 | step 400: blog 1.156, wiki 1.003


mix  lr 1e-05 seed 0 (166 s): step 0: blog 1.163, wiki 0.981 | step 50: blog 1.096, wiki 0.992 | step 100: blog 1.081, wiki 0.995 | step 200: blog 1.070, wiki 1.003 | step 400: blog 1.055, wiki 1.008


hard lr 1e-05 seed 1 (157 s): step 0: blog 1.163, wiki 0.981 | step 50: blog 1.092, wiki 1.004 | step 100: blog 1.073, wiki 1.012 | step 200: blog 1.066, wiki 1.027 | step 400: blog 1.085, wiki 1.051


soft lr 1e-05 seed 1 (155 s): step 0: blog 1.163, wiki 0.981 | step 50: blog 1.166, wiki 0.992 | step 100: blog 1.163, wiki 0.993 | step 200: blog 1.163, wiki 0.999 | step 400: blog 1.160, wiki 1.005


mix  lr 1e-05 seed 1 (157 s): step 0: blog 1.163, wiki 0.981 | step 50: blog 1.098, wiki 0.991 | step 100: blog 1.082, wiki 0.993 | step 200: blog 1.070, wiki 1.000 | step 400: blog 1.059, wiki 1.007
teacher: blog 1.038, wiki 0.873 bits per byte; its top 64 tokens hold 92.5% of its probability on average
student before training: blog 1.163, wiki 0.981 bits per byte

student, bits per byte after each step, seed 0 then seed 1
labels          step 50       step 100      step 200      step 400
hard      blog  1.086 1.092   1.072 1.073   1.071 1.066   1.069 1.085
hard      wiki  1.002 1.004   1.013 1.012   1.031 1.027   1.048 1.051
soft      blog  1.166 1.166   1.165 1.163   1.161 1.163   1.156 1.160
soft      wiki  0.991 0.992   0.995 0.993   0.998 0.999   1.003 1.005
half each blog  1.096 1.098   1.081 1.082   1.070 1.070   1.055 1.059
half each wiki  0.992 0.991   0.995 0.993   1.003 1.000   1.008 1.007


The post printed:

```text
teacher: blog 1.038, wiki 0.873 bits per byte; its top 64 tokens hold 92.5% of its probability on average
student before training: blog 1.163, wiki 0.981 bits per byte

student, bits per byte after each step, seed 0 then seed 1
labels          step 50       step 100      step 200      step 400
hard      blog  1.086 1.092   1.072 1.072   1.069 1.065   1.068 1.084
hard      wiki  1.002 1.004   1.012 1.011   1.031 1.026   1.047 1.050
soft      blog  1.166 1.165   1.164 1.163   1.161 1.163   1.156 1.161
soft      wiki  0.991 0.991   0.995 0.993   0.998 0.999   1.003 1.005
half each blog  1.096 1.098   1.081 1.083   1.070 1.070   1.055 1.059
half each wiki  0.992 0.991   0.995 0.993   1.002 1.000   1.008 1.007
```

## `kl_trace.py`

Runs `teacher_val.py` (the teacher's top 64 on the held-out blog, `teacher_val.pt`), then `distill.py` for seed 0 of each kind of label with the KL to the teacher logged (`distill_trace.log`), then prints the table with `trace_table.py`.

In [6]:
!{sys.executable} kl_trace.py

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (531066 > 131072). Running this sequence through the model will result in indexing errors


soft lr 1e-05 seed 0 (180 s): step 0: blog 1.163, wiki 0.981, KL to teacher 0.367 | step 50: blog 1.166, wiki 0.991, KL to teacher 0.386 | step 100: blog 1.164, wiki 0.995, KL to teacher 0.390 | step 200: blog 1.161, wiki 0.998, KL to teacher 0.378 | step 400: blog 1.156, wiki 1.003, KL to teacher 0.371


hard lr 1e-05 seed 0 (160 s): step 0: blog 1.163, wiki 0.981, KL to teacher 0.367 | step 50: blog 1.086, wiki 1.002, KL to teacher 0.506 | step 100: blog 1.072, wiki 1.013, KL to teacher 0.581 | step 200: blog 1.071, wiki 1.031, KL to teacher 0.666 | step 400: blog 1.069, wiki 1.047, KL to teacher 0.783


mix  lr 1e-05 seed 0 (157 s): step 0: blog 1.163, wiki 0.981, KL to teacher 0.367 | step 50: blog 1.096, wiki 0.992, KL to teacher 0.401 | step 100: blog 1.081, wiki 0.995, KL to teacher 0.414 | step 200: blog 1.070, wiki 1.003, KL to teacher 0.418 | step 400: blog 1.055, wiki 1.008, KL to teacher 0.441


seed 0, held-out blog: bits per byte / KL to the teacher over its top 64 (nats)
labels             step 0        step 50       step 100       step 200       step 400
soft        1.163 / 0.367  1.166 / 0.386  1.164 / 0.390  1.161 / 0.378  1.156 / 0.371
hard        1.163 / 0.367  1.086 / 0.506  1.072 / 0.581  1.071 / 0.666  1.069 / 0.783
half each   1.163 / 0.367  1.096 / 0.401  1.081 / 0.414  1.070 / 0.418  1.055 / 0.441


The post printed:

```text
seed 0, held-out blog: bits per byte / KL to the teacher over its top 64 (nats)
labels             step 0        step 50       step 100       step 200       step 400
soft        1.163 / 0.367  1.166 / 0.386  1.164 / 0.390  1.161 / 0.378  1.156 / 0.371
hard        1.163 / 0.367  1.086 / 0.505  1.072 / 0.578  1.069 / 0.662  1.068 / 0.781
half each   1.163 / 0.367  1.096 / 0.401  1.081 / 0.413  1.070 / 0.418  1.055 / 0.441
```

## `soft_check_both.py`

Runs `soft_check.py` at learning rates 1e-5 and 1e-4: 100 steps on soft labels, the training loss split into the KL over the teacher's top 64 and the student's mass outside them.

In [7]:
!{sys.executable} soft_check_both.py

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (531066 > 131072). Running this sequence through the model will result in indexing errors


lr 1e-05: held-out KL to teacher 0.367
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


  steps   1- 10: training KL 0.323, -log mass on top 64 0.150


  steps  11- 20: training KL 0.337, -log mass on top 64 0.091


  steps  21- 30: training KL 0.344, -log mass on top 64 0.090


...

  steps  61- 70: training KL 0.588, -log mass on top 64 0.149


  steps  71- 80: training KL 0.609, -log mass on top 64 0.157


  steps  81- 90: training KL 0.606, -log mass on top 64 0.158


  steps  91-100: training KL 0.576, -log mass on top 64 0.136


  after 100 steps: held-out KL to teacher 0.701


The post printed:

```text
lr 1e-05: held-out KL to teacher 0.367
  steps   1- 10: training KL 0.323, -log mass on top 64 0.150
  steps  11- 20: training KL 0.337, -log mass on top 64 0.091
  steps  21- 30: training KL 0.344, -log mass on top 64 0.091
  steps  31- 40: training KL 0.358, -log mass on top 64 0.105
  steps  41- 50: training KL 0.323, -log mass on top 64 0.084
  steps  51- 60: training KL 0.323, -log mass on top 64 0.087
  steps  61- 70: training KL 0.317, -log mass on top 64 0.090
  steps  71- 80: training KL 0.327, -log mass on top 64 0.095
  steps  81- 90: training KL 0.337, -log mass on top 64 0.090
  steps  91-100: training KL 0.296, -log mass on top 64 0.080
  after 100 steps: held-out KL to teacher 0.390
lr 0.0001: held-out KL to teacher 0.367
  steps   1- 10: training KL 0.367, -log mass on top 64 0.111
  steps  11- 20: training KL 0.439, -log mass on top 64 0.100
  steps  21- 30: training KL 0.579, -log mass on top 64 0.134
  steps  31- 40: training KL 0.600, -log mass on top 64 0.153
  steps  41- 50: training KL 0.563, -log mass on top 64 0.133
  steps  51- 60: training KL 0.616, -log mass on top 64 0.141
  steps  61- 70: training KL 0.587, -log mass on top 64 0.149
  steps  71- 80: training KL 0.606, -log mass on top 64 0.158
  steps  81- 90: training KL 0.608, -log mass on top 64 0.157
  steps  91-100: training KL 0.566, -log mass on top 64 0.135
  after 100 steps: held-out KL to teacher 0.678
```